In [ ]:
%pip install --upgrade openai pydantic mermaid-py

# Deep Research Agent

In this notebook you will learn how to use OpenAI's deep research models (`o3-deep-research` / `o4-mini-deep-research`) via the Responses API to conduct comprehensive, multi-source research that browses the web, analyzes data, and synthesizes detailed research-analyst-level reports.

## What is Deep Research?

Deep research models are a specialized class of OpenAI reasoning models designed to act as autonomous research analysts. Unlike standard chat models that respond in seconds, deep research models spend **minutes** actively browsing the web, reading documents, and synthesizing information from hundreds of sources.

### Key Capabilities

- **Web browsing**: Deep research models autonomously search the web, follow links, read full articles, and extract relevant information across dozens of queries.
- **File search**: They can search through uploaded files and documents to incorporate private data into their analysis.
- **Code analysis**: When paired with the code interpreter tool, they can perform quantitative analysis, create comparisons, and process data.
- **Multi-source synthesis**: The models cross-reference information across many sources to produce comprehensive, well-cited reports.

### Available Models

| Model | Description | Best For |
|-------|-------------|----------|
| `o3-deep-research` | Highest quality deep research | Complex analyses requiring maximum accuracy |
| `o4-mini-deep-research` | Cost-effective deep research | Most research tasks, good quality at lower cost |

### Use Cases

- **Market analysis**: Gather competitive data, pricing information, and market trends from multiple sources.
- **Legal research**: Survey case law, regulatory changes, and compliance requirements.
- **Competitive intelligence**: Compare products, features, and strategies across competitors.
- **Scientific literature review**: Survey research papers, findings, and methodologies in a field.

### Important Considerations

- **Execution time**: Deep research requests take **2-10 minutes** (or more) to complete. This is by design; the model is doing real research work.
- **Cost**: Deep research is significantly more expensive than standard API calls, so use it strategically for tasks that genuinely require multi-source synthesis.
- **Background mode**: Because of the long execution times, you should use background mode to avoid timeout issues.

## Setup

In [ ]:
import os
import json
import time
from getpass import getpass
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

# Deep research can take several minutes - set a longer timeout
client = OpenAI(timeout=600)

MODEL = "gpt-5-mini"
DEEP_RESEARCH_MODEL = "o4-mini-deep-research"  # Cost-effective deep research
# Alternative: "o3-deep-research" for highest quality

## Part 1: Basic Deep Research

The simplest way to use deep research is a single `responses.create` call with the `web_search_preview` tool. The model will autonomously decide what to search for, which pages to read, and how to synthesize the information.

> **Warning**: The cell below will take **2-10 minutes** to complete. Deep research models perform extensive web browsing and analysis before returning results. This is completely normal, so sit tight!

In [ ]:
response = client.responses.create(
    model=DEEP_RESEARCH_MODEL,
    input="""Research the current state of AI agent frameworks in 2026.
    
Include:
- Major frameworks and their key features
- Adoption trends and community size
- Strengths and weaknesses of each
- Recommendations for different use cases

Prioritize reliable sources: official documentation, GitHub repositories, 
technical blogs from recognized experts.""",
    tools=[
        {"type": "web_search_preview"}
    ]
)

# Add at top of cell: from IPython.display import display
display(Markdown(response.output_text))

## Part 2: Background Mode for Deep Research

For production use, **background mode** is the recommended approach for deep research. Here is why:

- **Timeout resilience**: Deep research requests can take many minutes. Background mode prevents HTTP timeouts and connection drops from losing your results.
- **Polling or webhooks**: You can poll for results at your own pace, or configure webhooks to be notified when research completes.
- **Connection independence**: Once launched, the research continues on OpenAI's servers even if your connection drops.

The pattern is:
1. Launch the request with `background=True`
2. Receive an immediate response with a response ID and `status` (usually `in_progress`)
3. Poll `client.responses.retrieve(id)` until `status == "completed"`

In [ ]:
# Launch research in background mode
response = client.responses.create(
    model=DEEP_RESEARCH_MODEL,
    input="""Conduct a detailed analysis of the economic impact of large language models 
on the software development industry in 2025-2026.

Requirements:
- Include specific statistics and revenue figures
- Compare productivity gains reported by different organizations
- Analyze the impact on different developer roles
- Include data on AI coding tool adoption rates
- Cite all sources with URLs

Be analytical and data-driven. Each claim should be supported by evidence.""",
    background=True,
    tools=[
        {"type": "web_search_preview"}
    ]
)

print(f"Research started! Response ID: {response.id}")
print(f"Status: {response.status}")

In [ ]:
# Poll for completion
while response.status != "completed":
    time.sleep(10)  # Check every 10 seconds
    response = client.responses.retrieve(response.id)
    print(f"Status: {response.status}...")

print(f"\nResearch complete!")
print(f"Output length: {len(response.output_text)} characters")

In [ ]:
# Display the full research report
# Add at top of cell: from IPython.display import display
display(Markdown(response.output_text))

## Inspecting the Research Process

The `response.output` array contains all the steps the model took during its research, including every web search query and the final synthesized message. Inspecting this is useful for:

- **Understanding what the model searched for**: See the exact queries it used.
- **Verifying source quality**: Check which URLs it cited.
- **Debugging**: If results are not what you expected, the search queries reveal what the model focused on.

In [ ]:
print(f"Total output items: {len(response.output)}")
print()

search_count = 0
for item in response.output:
    if item.type == "web_search_call":
        search_count += 1
        if hasattr(item, 'action') and hasattr(item.action, 'query'):
            print(f"  Search #{search_count}: {item.action.query}")
        elif hasattr(item, 'action') and hasattr(item.action, 'type'):
            print(f"  Action #{search_count}: {item.action.type}")

print(f"\nTotal web searches performed: {search_count}")

# Extract citations from the final message
for item in response.output:
    if item.type == "message":
        for content in item.content:
            if hasattr(content, 'annotations') and content.annotations:
                print(f"\nCitations found: {len(content.annotations)}")
                for ann in content.annotations[:10]:  # Show first 10
                    print(f"  - {ann.title}: {ann.url}")

## Part 3: Controlling Cost with `max_tool_calls`

Deep research models can make dozens (even hundreds) of tool calls during a single request: web searches, page reads, code executions. Each tool call adds to the cost.

The `max_tool_calls` parameter lets you set an upper bound on how many tool calls the model can make. This is useful for:

- **Budget control**: Prevent unexpectedly expensive requests.
- **Faster results**: Fewer searches means faster completion.
- **Scoped research**: When you need a quick overview rather than an exhaustive report.

> **Tip**: A good starting point is `max_tool_calls=10` for focused research, or leave it unlimited for comprehensive analysis.

In [ ]:
# Constrained research - fewer searches, lower cost, faster completion
response = client.responses.create(
    model=DEEP_RESEARCH_MODEL,
    input="What are the top 3 trends in AI safety research in 2026?",
    tools=[{"type": "web_search_preview"}],
    max_tool_calls=10  # Limit to 10 tool calls
)

# Add at top of cell: from IPython.display import display
display(Markdown(response.output_text))

## Part 4: Prompt Enrichment Pattern

A powerful pattern recommended by OpenAI is **prompt enrichment**: using a fast, inexpensive model to rewrite a user's question into detailed research instructions before sending it to the deep research model.

This three-step pattern works as follows:

1. **User submits a question**: A short, natural-language question (e.g., "How are companies using AI agents?").
2. **Fast model enriches the prompt**: A model like `gpt-5-mini` rewrites it into structured research instructions with specific aspects to investigate, source priorities, and expected output format.
3. **Deep research executes the enriched prompt**: The deep research model receives detailed, well-structured instructions, producing better results.

This pattern is especially useful when:
- End users submit vague or short questions
- You want consistent, structured research output
- You want to add domain-specific guidance without modifying the user's question

In [ ]:
def enrich_and_research(user_question):
    """
    Two-phase approach:
    1. Use a fast model to enrich the prompt into detailed research instructions
    2. Send the enriched prompt to deep research for comprehensive analysis
    """
    # Phase 1: Enrich the prompt with a fast model
    enrichment = client.responses.create(
        model=MODEL,
        instructions="""You will be given a research question. Rewrite it as detailed 
instructions for a research analyst. Include:
1. Specific aspects to investigate
2. Types of sources to prioritize
3. Expected output format (report with sections)
4. Any relevant context or constraints

Do NOT conduct the research yourself. Just provide instructions.""",
        input=user_question
    )

    enriched_prompt = enrichment.output_text
    print("Enriched prompt:")
    print(enriched_prompt)
    print("\n" + "=" * 60 + "\n")

    # Phase 2: Deep research with the enriched prompt
    response = client.responses.create(
        model=DEEP_RESEARCH_MODEL,
        input=enriched_prompt,
        tools=[{"type": "web_search_preview"}]
    )

    return response


# Example usage
response = enrich_and_research("How are companies using AI agents in production?")
# Add at top of cell: from IPython.display import display
display(Markdown(response.output_text))

## Part 5: Deep Research with Code Interpreter

Deep research models can use multiple tools simultaneously. By adding the `code_interpreter` tool alongside `web_search_preview`, you enable the model to:

- Perform calculations on data it finds during research
- Create structured comparisons and tables
- Analyze numerical trends
- Generate charts and visualizations (returned as files)

This is particularly useful for research tasks that involve quantitative analysis.

> **Note**: Adding the code interpreter increases the potential cost and execution time, since the model may run multiple code executions during its research.

In [ ]:
response = client.responses.create(
    model=DEEP_RESEARCH_MODEL,
    input="""Research and analyze the growth of open-source AI projects on GitHub.

Tasks:
1. Find data on the most popular AI/ML repositories
2. Analyze trends in stars, contributors, and release frequency
3. Create a comparison of the top 5 projects
4. Include any quantitative analysis you can perform

Use the code interpreter to perform calculations and create structured comparisons.""",
    tools=[
        {"type": "web_search_preview"},
        {"type": "code_interpreter", "container": {"type": "auto"}}
    ]
)

# Add at top of cell: from IPython.display import display
display(Markdown(response.output_text))

## When to Use Deep Research

Deep research is a powerful but expensive tool. Here is a guide for when it is worth the cost:

| Use Case | Recommended? | Why |
|----------|:------------:|-----|
| Quick factual questions | No | Use `gpt-5-mini` with `web_search_preview` instead; it is faster and cheaper |
| Comprehensive market analysis | **Yes** | Needs synthesis across many sources, competitor sites, reports |
| Competitive intelligence | **Yes** | Requires multi-source comparison and cross-referencing |
| Scientific literature review | **Yes** | Deep browsing, reading papers, and synthesizing findings |
| Legal/regulatory research | **Yes** | Requires careful reading of multiple regulatory documents |
| Real-time data needs | No | Too slow for real-time because deep research takes minutes |
| Simple summaries | No | Overkill. Use cheaper models for straightforward tasks |
| Code generation | No | Use reasoning models like `o3` or `o4-mini` directly |

### Cost Optimization Tips

1. **Use `o4-mini-deep-research`** for most tasks. It is significantly cheaper than `o3-deep-research` while still delivering high-quality results.
2. **Set `max_tool_calls`** to limit the number of searches when you need focused rather than exhaustive research.
3. **Use prompt enrichment** to ensure the model gets clear instructions on the first try, reducing wasted searches.
4. **Cache results** for repeated queries. Deep research results do not change significantly over short time periods.

## Your Turn: Compare Two Topics Side by Side

Now put what you learned to work! You will build a comparison research agent that takes two topics, researches both using `web_search_preview`, and produces a structured side-by-side comparison.

In [ ]:
# Model for general-purpose tasks in the exercise
# (Deep research models are specialized; for query generation and synthesis we use gpt-5-mini)
MODEL = "gpt-5-mini"

In [ ]:
class ComparisonRow(BaseModel):
    dimension: str = Field(description="What is being compared (e.g. performance, community size)")
    topic_a: str = Field(description="Finding for the first topic")
    topic_b: str = Field(description="Finding for the second topic")


class ComparisonReport(BaseModel):
    topic_a_name: str = Field(description="Name of the first topic")
    topic_b_name: str = Field(description="Name of the second topic")
    rows: list[ComparisonRow] = Field(description="Comparison rows, one per dimension")
    overall_summary: str = Field(description="Overall summary of the comparison")

### Now it is your turn to build the `compare_topics` function!

Fill in the function body below. A suggested approach:

1. Search the web for information about `topic_a` (`num_searches` times).
2. Search the web for information about `topic_b` (`num_searches` times).
3. Combine all findings and ask the model to produce a `ComparisonReport`
   using structured output.

Feel free to use any approach you like. The important thing is that the
function returns a valid `ComparisonReport`.

In [ ]:
def compare_topics(topic_a: str, topic_b: str, num_searches: int = 2) -> ComparisonReport:
    """Research two topics and produce a structured side-by-side comparison."""

    # YOUR CODE HERE
    # Suggested steps:

    # Step 1: Search for topic_a
    # Run num_searches web searches about topic_a and collect the findings.
    # You can either:
    #   a) Generate targeted queries first, then search each one, OR
    #   b) Do a single search per topic with a detailed prompt.

    # Step 2: Search for topic_b
    # Same approach as step 1, but for the second topic.

    # Step 3: Synthesize into a ComparisonReport
    # Build a prompt that includes findings for both topics.
    # Use text={"format": comparison_schema} for structured output.
    # Parse with ComparisonReport.model_validate_json(response.output_text)

    # comparison_schema = {
    #     "type": "json_schema",
    #     "name": "comparison_report",
    #     "strict": True,
    #     "schema": ComparisonReport.model_json_schema()
    # }

    pass  # Remove this line when you add your implementation

### Test your comparison agent

Run the cell below to compare two AI frameworks. If your implementation
is correct, you will see a formatted comparison table and summary.

In [ ]:
report = compare_topics("LangChain", "LlamaIndex", num_searches=2)

print(f"Comparing: {report.topic_a_name} vs {report.topic_b_name}")
print("=" * 70)
print()
print(f"{'Dimension':<25} {'  ' + report.topic_a_name:<22} {'  ' + report.topic_b_name:<22}")
print("-" * 70)
for row in report.rows:
    print(f"{row.dimension:<25} {row.topic_a:<22} {row.topic_b:<22}")
print()
print("Overall Summary:")
print(report.overall_summary)

## Summary

Great work making it through this notebook! Here are the key takeaways:

- **Deep research models** (`o3-deep-research`, `o4-mini-deep-research`) conduct multi-step, autonomous research across hundreds of web sources, producing research-analyst-level reports.
- **Always use background mode** for reliability. Deep research requests take minutes, and background mode handles timeouts and connectivity issues gracefully.
- **Control costs with `max_tool_calls`** to limit the number of web searches and tool invocations the model performs.
- **Enrich user prompts** before sending them to deep research. A fast model like `gpt-5-mini` can rewrite vague questions into detailed research instructions for better results.
- **Inspect the output array** to see exactly what searches and analysis the model performed, including all queries and cited sources.
- **Combine tools strategically**: `web_search_preview` for browsing, `code_interpreter` for quantitative analysis, and `file_search` for private documents.
- **Use deep research selectively.** It is powerful but expensive and slow, so reserve it for tasks that genuinely require multi-source synthesis and comprehensive analysis.